### Initial Setup

In [ ]:
# importing libraries
from mcts import *
from db_management import *
from gui import *
import time

# parameters we'll use
database_location = 'db/Connect_4.db'
neural_network_id = 17 # change this to the neural network we want to train
max_iterations = 100 # number of monte carlo tree search simulations
max_depth = 8 # how deep MCTS will search from the current root node
exploration_constant = 3
num_self_play_games = 20 # number of games per training batch
learning_rate = 0.15
num_epochs = 200 # number of epochs during the learning stage (i.e. actually updating the neural network)
regularization = 'L2' # ridge regression
lambda_const = 0.05
sampling_rate = 0.2 # if set less than 1, then we will be using mini-batch gradient descent

In [ ]:
# query we'll use for training
recent_game_states = 1000 # number of recent game states that will be used for training
recent_end_game_states = 200 # number of end game states that will be used for training (might overlap with recent game states)

training_query = f"""
with end_games as (
    select
        a.*,
        row_number() over(
            partition by game_hist_id
            order by turn_number desc
        ) as ranking
    from game_turns a
    where game_hist_id not between 421 and 620
),
batch_games as (
    select game_hist_id
    from games_history
    where 
        creation_date > '2026-02-06'
        and game_hist_id not between 421 and 620
)
-- getting end game states from recent games
select *
from (
    select
        game_hist_id,
        turn_number,
        game_state,
        mcts_visit_ratios,
        value_head
    from end_games
    where ranking between 4 and 8
    order by
        game_hist_id desc,
        turn_number asc
    limit {recent_end_game_states}
)
union
-- getting the more recent game states
select *
from (
    select
        a.game_hist_id,
        a.turn_number,
        a.game_state,
        a.mcts_visit_ratios,
        a.value_head
    from game_turns a
    inner join batch_games b
        on a.game_hist_id = b.game_hist_id
    order by a.turn_id desc
    limit {recent_game_states}
)
order by
    game_hist_id desc,
    turn_number desc
"""

In [3]:
# testing the training query

query_result = []

with sqlite3.connect(database_location) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute(training_query)
    rows = cursor.fetchall()

    for i in range(len(rows)):
        query_result.append((rows[i]['game_hist_id'], rows[i]['turn_number'], rows[i]['game_state'], rows[i]['mcts_visit_ratios'], rows[i]['value_head']))

print(f'query length = {len(query_result)}')

query length = 1000


### Running Training Loop

In [ ]:
for i in range(41, 51): # LEFT number is the starting batch, RIGHT number is the ending batch (not inclusive)
    training_loop(
        database_location=database_location,
        neural_network_id=neural_network_id,
        batch_name=i,
        max_iterations=max_iterations,
        max_depth=max_depth,
        exploration_constant=exploration_constant,
        num_self_games=num_self_play_games,
        training_query=training_query,
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        regularization=regularization,
        lambda_const=lambda_const,
        sampling_rate=sampling_rate
    )

Batch 41 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 42 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 43 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 44 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 45 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 46 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 47 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 48 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
Batch 49 - Game 1==>2==>3==>4==>5==>6==>7==>8==>9==>10==>11==>12==>13==>14==>15==>16==>17==>18==>19==>20==>end
B